# 04a — Primary ICU-COPD survival analysis

Run `sql/04_build_outcomes.sql` and then `sql/04_build_outcomes_extended.sql` before this notebook. The shared loader reads the extended outcome table and detailed prescription-order flags; this notebook refreshes models and figures, not BigQuery tables.

Age is the admission-year estimate; duration is continuous elapsed days. The refined primary model adjusts for age and sex and stratifies by anchor-year group, without first-day SOFA. Live discharge is censored. SOFA-containing outputs refreshed by the shared workflow are separate exploratory sensitivity analyses because they can use information through 24 hours after ICU follow-up starts.

The original primary model retains ordinary admission-level intervals and live-discharge censoring. Separate saved supplementary results address [patient-cluster inference](05a_patient_cluster_sensitivity.ipynb), [competing death/discharge endpoints](05b_competing_risks.ipynb) and a [24-hour landmark](05c_time_aligned_severity.ipynb); they do not replace the original main estimates.


<!-- Generated main figure guide: intro; edit scripts/render_main_figure_guides.py -->

## Saved clinical results

The tables and figures in this section display saved clinical execution results from [the public aggregate record](../docs/CURRENT_ANALYSIS.json). They do not represent a new clinical rerun. The separately labeled execution section describes how to reproduce outputs. Any synthetic execution is explicitly labeled and is not a clinical MIMIC result.

<!-- Generated main figure guide: km; edit scripts/figure_interpretation.py -->

### Figure 1 — Kaplan–Meier estimates

**Data source:** Saved clinical execution results; [aggregate record](../docs/CURRENT_ANALYSIS.json).

![Figure 1 — Kaplan–Meier estimates](../assets/kaplan_meier_curve.png)

**How to read**

- The horizontal axis is elapsed days since ICU entry; the vertical axis is the unadjusted Kaplan–Meier survival estimate. Blue represents qualifying pre-ICU RAAS orders and orange represents no qualifying orders. Shading shows pointwise 95% confidence intervals.
- Live discharge is censored. Consequently, one minus this curve is not the observed cumulative probability of hospital death when discharge is a competing event; use the separate CIF figure for that question.

**What the results show**

- The overall two-group log-rank p-value is 0.00417. It compares the unadjusted survival experience over follow-up, rather than testing every plotted day or controlling for age, severity and other confounding.

**Interpretation limits**

- Wider intervals in the late tail indicate less precise estimates. Visual separation is an observational association and does not establish a medication benefit. The curve is not an adjusted Cox prediction.

<!-- Generated main figure guide: runtime; edit scripts/render_main_figure_guides.py -->

## Runtime execution

The following cells run the shared refresh in an authorized BigQuery environment. They recompute the current models and exported figures; they are separate from the saved figures above.

In [ ]:
from pathlib import Path
import sys
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from current_analysis import refresh


In [ ]:
# Recompute current tables once; all figures, aggregate JSON and report source stay aligned.
results = refresh(REPO_ROOT)
print({"n": results["n"], "deaths": results["deaths"], "version": results["analysis_version"]})


In [ ]:
results["models"]["primary_refined"]["estimates"]


In [ ]:
results["models"]["primary_refined"]["ph_rank_p"]


In [ ]:
print("All four figures and current report source refreshed; render Quarto to update HTML.")
